In [7]:
from bs4 import SoupStrainer
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

triton_urls = [
    "https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/user_guide/model_repository.html",
    "https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/user_guide/model_management.html",
    "https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/protocol/README.html",
]

mlflow_urls = [
    "https://mlflow.org/docs/latest/self-hosting/architecture/tracking-server/",
    "https://mlflow.org/docs/latest/self-hosting/troubleshooting/",
    "https://mlflow.org/docs/latest/ml/deployment/",
    "https://mlflow.org/docs/latest/ml/deployment/deploy-model-locally",
    "https://mlflow.org/docs/latest/ml/model-registry/tutorial",
]

kubernetes_urls = [
    "https://kubernetes.io/docs/tasks/debug/debug-application/",
    "https://kubernetes.io/docs/tasks/debug/debug-application/debug-running-pod/",
    "https://kubernetes.io/docs/tasks/debug/debug-application/debug-init-containers/",
]

aws_lambda_urls = [
    "https://docs.aws.amazon.com/lambda/latest/dg/lambda-troubleshooting.html",
    "https://docs.aws.amazon.com/lambda/latest/dg/troubleshooting-invocation.html",
]

aws_ecs_urls = [
    "https://docs.aws.amazon.com/AmazonECS/latest/developerguide/troubleshooting.html",
    "https://docs.aws.amazon.com/AmazonECS/latest/developerguide/cannot-start-container.html",
    "https://docs.aws.amazon.com/AmazonECS/latest/developerguide/container-runtime-error.html",
]

fastapi_urls = [
    "https://fastapi.tiangolo.com/deployment/concepts/",
    "https://fastapi.tiangolo.com/deployment/server-workers/",
    "https://fastapi.tiangolo.com/deployment/docker/",
]


sources = {
    "triton": triton_urls,
    "mlflow": mlflow_urls,
    "kubernetes": kubernetes_urls,
    "aws_lambda": aws_lambda_urls,
    "aws_ecs": aws_ecs_urls,
    "fastapi": fastapi_urls,
}


# ==========================================
# Load Documentation
# ==========================================

all_docs = []
for technology, urls in sources.items():
    for url in urls:
        loader = WebBaseLoader(
            [url],
            bs_kwargs={
                "parse_only": SoupStrainer("main")
            }
        )
        docs = loader.load()

        for doc in docs:
            doc.metadata["technology"] = technology
            doc.metadata["source_url"] = url
            doc.metadata["document_type"] = "official_documentation"
            doc.metadata["title"] = doc.metadata.get(
                "title",
                f"{technology} documentation"
            )

        all_docs.extend(docs)


print(f"Total documents loaded: {len(all_docs)}")


splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)
chunks = splitter.split_documents(all_docs)

print(f"Total chunks: {len(chunks)}")


USER_AGENT environment variable not set, consider setting it to identify your requests.


Total documents loaded: 19
Total chunks: 267


In [ ]:
embedder = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedder,
    persist_directory="./my_chroma_db_v1"
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 8}
)

retriever

In [ ]:


# Load the same embedding model used when creating the DB
embedder = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

# Reconnect to your existing Chroma database
vectorstore = Chroma(
    persist_directory="./my_chroma_db_v1",
    embedding_function=embedder,
)

# Create retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 8}
)

print("Chroma DB loaded successfully!")
print(retriever)


/Users/raghuvar/Desktop/vsc/final-rag-project/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4522.52it/s]


Chroma DB loaded successfully!
tags=['Chroma', 'HuggingFaceEmbeddings'] vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x103b77a90> search_kwargs={'k': 8}


In [4]:
response = retriever.invoke(
    "Why is my Triton model returning HTTP 503?"
)

for i, doc in enumerate(response):

    print(f"\n========== RESULT {i + 1} ==========")

    print("Technology:", doc.metadata.get("technology"))
    print("Title:", doc.metadata.get("title"))
    print("Source:", doc.metadata.get("source_url"))

    print("\nContent:")
    print(doc.page_content)


========== RESULT 1 ==========
Technology: triton
Title: triton documentation
Source: https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/protocol/README.html

Content:
TRITONSERVER_ERROR_UNAVAILABLE
503
Service Unavailable

TRITONSERVER_ERROR_UNSUPPORTED
501
Not Implemented

TRITONSERVER_ERROR_UNKNOWN,TRITONSERVER_ERROR_INVALID_ARG,TRITONSERVER_ERROR_ALREADY_EXISTS,TRITONSERVER_ERROR_CANCELLED
400
Bad Request (default for other errors)

















 On this page
  


Restricted Protocols
IPv6
Mapping Triton Server Error Codes to HTTP Status Codes

========== RESULT 2 ==========
Technology: triton
Title: triton documentation
Source: https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/protocol/README.html

Content:
And can be tested via curl, for example:
$ curl -6 --verbose "http://[::1]:8000/v2/health/ready"
*   Trying ::1:8000...
* TCP_NODELAY set
* Connected to ::1 (::1) port 8000 (#0)
> GET /v2/health/ready HTTP/1.1
> Host: [::

In [5]:
response = retriever.invoke(
    "Why is my Kubernetes pod stuck in CrashLoopBackOff?"
)

for i, doc in enumerate(response):

    print(f"\n========== RESULT {i + 1} ==========")

    print("Technology:", doc.metadata.get("technology"))
    print("Title:", doc.metadata.get("title"))
    print("Source:", doc.metadata.get("source_url"))

    print("\nContent:")
    print(doc.page_content)


========== RESULT 1 ==========
Technology: kubernetes
Title: kubernetes documentation
Source: https://kubernetes.io/docs/tasks/debug/debug-application/debug-running-pod/

Content:
- containerID: containerd://5403af59a2b46ee5a23fb0ae4b1e077f7ca5c5fb7af16e1ab21c00e0e616462a
    image: docker.io/library/nginx:latest
    imageID: docker.io/library/nginx@sha256:2834dc507516af02784808c5f48b7cbe38b8ed5d0f4837f16e78d00deb7e7767
    lastState: {}
    name: nginx
    ready: true
    restartCount: 0
    started: true
    state:
      running:
        startedAt: "2022-02-17T21:51:05Z"
  hostIP: 192.168.0.113
  phase: Running
  podIP: 10.88.0.3
  podIPs:
  - ip: 10.88.0.3
  - ip: 2001:db8::1
  qosClass: Guaranteed
  startTime: "2022-02-17T21:51:01Z"
Examining pod logsFirst, look at the logs of the affected container:kubectl logs ${POD_NAME} -c ${CONTAINER_NAME}
If your container has previously crashed, you can access the previous container's crash log with:kubectl logs ${POD_NAME} -c ${CONTAINER_N

In [8]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever=BM25Retriever.from_documents(chunks)
bm25_retriever.k=8

In [9]:
bm25_retriever.invoke("why is my kubernetes container failing")

[Document(metadata={'source': 'https://fastapi.tiangolo.com/deployment/docker/', 'technology': 'fastapi', 'source_url': 'https://fastapi.tiangolo.com/deployment/docker/', 'document_type': 'official_documentation', 'title': 'fastapi documentation'}, page_content='⛔️ Shell form:\n# ⛔️ Don\'t do this\nCMD fastapi run app/main.py --port 80\n\nMake sure to always use the exec form to ensure that FastAPI can shutdown gracefully and lifespan events are triggered.\nYou can read more about it in the Docker docs for shell and exec form.\nThis can be quite noticeable when using docker compose. See this Docker Compose FAQ section for more technical details: Why do my services take 10 seconds to recreate or stop?.\nDirectory Structure¶\nYou should now have a directory structure like:\n.\n├── app\n│\xa0\xa0 ├── __init__.py\n│   └── main.py\n├── Dockerfile\n└── requirements.txt\n\nBehind a TLS Termination Proxy¶\nIf you are running your container behind a TLS Termination Proxy (load balancer) like Ng

In [10]:
from langchain_classic.retrievers import EnsembleRetriever

ensemble_retriever=EnsembleRetriever(
    retrievers=[retriever, bm25_retriever],
    weights=[0.6, 0.4]
)

In [11]:
response=ensemble_retriever.invoke("why is my kubernetes container failing")
for i, doc in enumerate(response):
    print("-----------------------------------------")
    print(f"Document {i}:")
    print(doc.page_content)
    print(doc.metadata)
    print("-----------------------------------------")

-----------------------------------------
Document 0:
NAME                                READY     STATUS    RESTARTS   AGE
nginx-deployment-1006230814-6winp   1/1       Running   0          7m
nginx-deployment-1006230814-fmgu3   1/1       Running   0          7m
nginx-deployment-1370807587-6ekbw   1/1       Running   0          1m
nginx-deployment-1370807587-fg172   0/1       Pending   0          1m
nginx-deployment-1370807587-fz9sd   0/1       Pending   0          1m
To find out why the nginx-deployment-1370807587-fz9sd pod is not running, we can use kubectl describe pod on the pending Pod and look at its events:kubectl describe pod nginx-deployment-1370807587-fz9sd
  Name:		nginx-deployment-1370807587-fz9sd
  Namespace:	default
  Node:		/
  Labels:		app=nginx,pod-template-hash=1370807587
  Status:		Pending
  IP:
  Controllers:	ReplicaSet/nginx-deployment-1370807587
  Containers:
    nginx:
      Image:	nginx
      Port:	80/TCP
      QoS Tier:
        memory:	Guaranteed
        cpu:

In [12]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [13]:
from sentence_transformers import CrossEncoder

cross_encoder=CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6447.67it/s]


In [14]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate

llm=init_chat_model(model_provider="groq", model="openai/gpt-oss-120b")

def reranker(query, top_k=5):
    docs=ensemble_retriever.invoke(query)

    # Create (query, document) pairs
    pairs=[
        (query, doc.page_content)
        for doc in docs
    ]
    # Get relevance scores
    scores=cross_encoder.predict(pairs, batch_size=8)

    # Attach scores to documents
    scored_docs=list(zip(docs, scores))

    # Sort by relevance score
    scored_docs = sorted(
        scored_docs,
        key=lambda x: x[1],
        reverse=True
    )

    return [
        (doc, float(score))
        for doc, score in scored_docs[:top_k]
    ]

In [15]:
import psutil

ram = psutil.virtual_memory()

print(f"RAM used: {ram.percent}%")
print(f"Available: {ram.available / 1024**3:.2f} GB")


RAM used: 80.1%
Available: 1.59 GB


In [16]:
response=reranker("why is my kubernetes container failing")
for i, (doc, score) in enumerate(response):
    print(f"\n========== RESULT {i + 1} ==========")
    print(f"Score: {score:.4f}")
    print(f"Technology: {doc.metadata.get('technology')}")
    print(f"Title: {doc.metadata.get('title')}")
    print(f"Source: {doc.metadata.get('source_url')}")
    print("\nContent:")
    print(doc.page_content)


========== RESULT 1 ==========
Score: 4.2446
Technology: kubernetes
Title: kubernetes documentation
Source: https://kubernetes.io/docs/tasks/debug/debug-application/

Content:
Kubernetes DocumentationTasksMonitoring, Logging, and DebuggingTroubleshooting ApplicationsTroubleshooting ApplicationsDebugging common containerized application issues.This doc contains a set of resources for fixing issues with containerized applications. It covers things like common issues with Kubernetes resources (like Pods, Services, or StatefulSets), advice on making sense of container termination messages, and ways to debug running containers.Debug PodsDebug ServicesDebug a StatefulSetDetermine the Reason for Pod FailureDebug Init ContainersDebug Running PodsGet a Shell to a Running ContainerFeedbackWas this page helpful?Yes
NoThanks for the feedback. If you have a specific, answerable question about how to use Kubernetes, ask it on
Stack Overflow.
Open an issue in the GitHub Repository if you want to
rep

In [17]:
response=reranker("Why is my Triton model returning HTTP 503?")
for i, (doc, score) in enumerate(response):
    print(f"\n========== RESULT {i + 1} ==========")
    print(f"Score: {score:.4f}")
    print(f"Technology: {doc.metadata.get('technology')}")
    print(f"Title: {doc.metadata.get('title')}")
    print(f"Source: {doc.metadata.get('source_url')}")
    print("\nContent:")
    print(doc.page_content)


========== RESULT 1 ==========
Score: 3.1712
Technology: triton
Title: triton documentation
Source: https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/user_guide/model_management.html

Content:
Model Management










Model Management#
Triton provides model management APIs are part of the HTTP/REST and GRPC protocols, and as part of the C API.
Triton operates in one of three model control modes: NONE, EXPLICIT, or POLL.
The model control mode determines how changes to the model repository are handled by Triton and which of these protocols and APIs are available.

Model Control Mode NONE#

Triton attempts to load all models in the model repository at startup.
Models that Triton is not able to load will be marked as “UNAVAILABLE” and will not be available for inferencing.
Changes to the model repository while the server is running will be ignored.
Model load and unload requests using the model control protocol will have no affect and will return an error res

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_classic.retrievers import EnsembleRetriever
from sentence_transformers import CrossEncoder
from langchain_core.prompts import PromptTemplate

def rag(state):
    embedder = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-mpnet-base-v2"
    )
    
    # Reconnect to your existing Chroma database
    vectorstore = Chroma(
        persist_directory="./my_chroma_db_v1",
        embedding_function=embedder,
    )

    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 8}
    )
    print("Chroma DB loaded successfully")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150
    )

    # how will all docs come ?
    chunks = splitter.split_documents(all_docs)
    bm25_retriever=BM25Retriever.from_documents(chunks)
    bm25_retriever.k=8

    ensemble_retriever=EnsembleRetriever(
        retrievers=[retriever, bm25_retriever],
        weights=[0.6, 0.4]
    )

    cross_encoder=CrossEncoder(
        "cross-encoder/ms-marco-MiniLM-L6-v2"
    )

    top_k=5
    query=state['query'] # varibale may change
    docs=ensemble_retriever.invoke(query)  

    # Create (query, document) pairs
    pairs=[
        (query, doc.page_content)
        for doc in docs
    ]
    # Get relevance scores
    scores=cross_encoder.predict(pairs, batch_size=8)

    # Attach scores to documents
    scored_docs=list(zip(docs, scores))

    # Sort by relevance score
    scored_docs = sorted(
        scored_docs,
        key=lambda x: x[1],
        reverse=True
    )

    output= [
        (doc, float(score))
        for doc, score in scored_docs[:top_k]
    ]

    # llm part
    prompt=PromptTemplate.from_template(
        """answer the user query based on the supplied context, 
        if the documents dont contain the answer just say I dont know the answer
        Query: {query}
        Context: {output}"""
    )

    chain=prompt | llm
    llm_response=chain.invoke({"query":query, "output":output})
    
    return {"response":llm_response}

## final node

In [20]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from sentence_transformers import CrossEncoder


# ==========================================
# Configuration
# ==========================================

CHROMA_PATH = "./my_chroma_db_v1"

EMBEDDING_MODEL = (
    "sentence-transformers/all-mpnet-base-v2"
)

CROSS_ENCODER_MODEL = (
    "cross-encoder/ms-marco-MiniLM-L6-v2"
)

DENSE_K = 8
BM25_K = 8
FINAL_K = 5

DENSE_WEIGHT = 0.6
BM25_WEIGHT = 0.4


# ==========================================
# Initialize Embeddings
# ==========================================

embedder = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL
)


# ==========================================
# Load Chroma
# ==========================================

vectorstore = Chroma(
    persist_directory=CHROMA_PATH,
    embedding_function=embedder
)

dense_retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": DENSE_K
    }
)


# ==========================================
# Initialize BM25
# ==========================================

bm25_retriever = BM25Retriever.from_documents(
    chunks
)

bm25_retriever.k = BM25_K


# ==========================================
# Hybrid Retriever
# ==========================================

ensemble_retriever = EnsembleRetriever(
    retrievers=[
        dense_retriever,
        bm25_retriever
    ],
    weights=[
        DENSE_WEIGHT,
        BM25_WEIGHT
    ]
)


# ==========================================
# Cross Encoder
# ==========================================

cross_encoder = CrossEncoder(
    CROSS_ENCODER_MODEL
)


# ==========================================
# Retrieval Function
# ==========================================

def retrieve_documents(
    query: str,
    top_k: int = FINAL_K
):

    # Hybrid retrieval
    docs = ensemble_retriever.invoke(query)

    if not docs:
        return []

    # Query-document pairs
    pairs = [
        (query, doc.page_content)
        for doc in docs
    ]

    # Cross encoder scoring
    scores = cross_encoder.predict(
        pairs,
        batch_size=8,
        show_progress_bar=False
    )

    # Attach scores
    scored_docs = list(
        zip(docs, scores)
    )

    # Sort by relevance
    scored_docs.sort(
        key=lambda x: x[1],
        reverse=True
    )

    # Build structured results
    results = []

    for doc, score in scored_docs[:top_k]:

        results.append({
            "content": doc.page_content,
            "score": float(score),
            "technology": doc.metadata.get(
                "technology"
            ),
            "title": doc.metadata.get(
                "title"
            ),
            "source_url": doc.metadata.get(
                "source_url"
            ),
            "document": doc
        })

    return results


# ==========================================
# LangGraph RAG Node
# ==========================================

def rag_node(state):

    query = state["query"]

    retrieved_documents = retrieve_documents(
        query=query,
        top_k=FINAL_K
    )

    return {
        "retrieved_documents": retrieved_documents
    }

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6926.52it/s]


In [21]:
state = {
    "query": "Why is my Triton model returning HTTP 503?"
}

result = rag_node(state)

for i, doc in enumerate(
    result["retrieved_documents"]
):

    print(
        f"\n========== RESULT {i + 1} =========="
    )

    print(
        "Score:",
        doc["score"]
    )

    print(
        "Technology:",
        doc["technology"]
    )

    print(
        "Title:",
        doc["title"]
    )

    print(
        "Source:",
        doc["source_url"]
    )

    print(
        "\nContent:",
        doc["content"]
    )


========== RESULT 1 ==========
Score: 3.1711671352386475
Technology: triton
Title: triton documentation
Source: https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/user_guide/model_management.html

Content: Model Management










Model Management#
Triton provides model management APIs are part of the HTTP/REST and GRPC protocols, and as part of the C API.
Triton operates in one of three model control modes: NONE, EXPLICIT, or POLL.
The model control mode determines how changes to the model repository are handled by Triton and which of these protocols and APIs are available.

Model Control Mode NONE#

Triton attempts to load all models in the model repository at startup.
Models that Triton is not able to load will be marked as “UNAVAILABLE” and will not be available for inferencing.
Changes to the model repository while the server is running will be ignored.
Model load and unload requests using the model control protocol will have no affect and will return 

In [ ]:
from rag.ingestion import save_documents_for_bm25

save_documents_for_bm25(chunks)

ModuleNotFoundError: No module named 'rag'